# LCProp bounded LC and PR notebook

Use a fresh hosted CPU session. Upload the exact wheel and fill in the
filename and SHA-256 in the setup cell. The batch remains a candidate for
hosted acceptance. Only the interface's Run button starts propagation.


In [ ]:
from pathlib import Path
import hashlib
import subprocess
import sys

wheel = Path("/content/REPLACE_WITH_WHEEL_FILENAME.whl")
expected_sha256 = "REPLACE_WITH_RECORDED_SHA256"
if any(name == "lcprop" or name.startswith("lcprop.") for name in sys.modules):
    raise RuntimeError("Use a fresh kernel to install the candidate.")
if hashlib.sha256(wheel.read_bytes()).hexdigest() != expected_sha256:
    raise ValueError("Wheel SHA-256 does not match the approved candidate.")
subprocess.run(
    [sys.executable, "-m", "pip", "install", "--force-reinstall", "--no-deps", str(wheel)],
    check=True,
)
subprocess.run(
    [sys.executable, "-m", "pip", "install", f"lcprop[colab] @ {wheel.as_uri()}"],
    check=True,
)


Restart the session after installation. Skip setup after restart. Enter the recorded checksum again below, then run the display cell.

In [ ]:
import json
from importlib import metadata
import subprocess
import sys

expected_sha256 = "REPLACE_WITH_RECORDED_SHA256"
dist = metadata.distribution("lcprop")
origin = json.loads(dist.read_text("direct_url.json") or "{}")
actual = origin.get("archive_info", {}).get("hashes", {}).get("sha256")
if actual is None or actual != expected_sha256:
    raise RuntimeError("The installed LCProp wheel does not match the candidate.")
subprocess.run([sys.executable, "-m", "pip", "check"], check=True)
for name in ("lcprop", "numpy", "scipy", "ipywidgets", "ipympl"):
    print(name, metadata.version(name))


In [ ]:
from google.colab import output
from IPython import get_ipython
from IPython.display import clear_output, display
from lcprop.notebook import InteractiveBatch

if "lcprop_batch" not in globals():
    output.enable_custom_widget_manager()
    get_ipython().run_line_magic("matplotlib", "widget")
    lcprop_batch = InteractiveBatch()
elif lcprop_batch.session.disposed:
    raise RuntimeError("Restart after saving needed files to replace a disposed controller.")
clear_output(wait=True)
display(lcprop_batch.widget)
